In [0]:
use catalog sky_dev;

In [0]:
-- Q01 KPI summary: net revenue, distinct bookings, active passengers, average booking value 
select
round(sum(ticket_price * (1 - discount)),2) as net_revenue,
count(distinct booking_id) as distinct_bookings,
count(distinct passenger_id) as active_passengers,
round(net_revenue / distinct_bookings,2) as average_booking_value
from gold.fact_booking

In [0]:
-- Q02 Monthly revenue trends
select month(booking_date) as month, year(booking_date) as year,
round(sum(ticket_price * (1 - discount)),2) as net_revenue,
round(sum(total_price),2)as total_nevenue
from gold.fact_booking
group by year, month
order by year, month

In [0]:
-- Q3 annual revenue with year over year change
select 
year(booking_date) as year,
round(sum(ticket_price * (1 - discount)),2) as net_revenue,
lag(round(sum((ticket_price * (1 - discount))),2)) over (order by year(booking_date)) as prev_year_revenue,
net_revenue - prev_year_revenue as yoy_change
from gold.fact_booking
group by year(booking_date)
order by year

In [0]:
-- Q4 revenue by passenger travel purpose
 select p.pa_travelpurpose as travel_purpose,
 round(sum(b.ticket_price * (1 - b.discount)),2) as revenue
 from gold.dim_passenger p
 join gold.fact_booking b 
 on p.pa_passengerid = b.passenger_id
 group by travel_purpose

In [0]:
-- Q05 top 10 passenger by net revenue
select passenger_id,
round(sum(ticket_price * (1 - discount)),2) as net_revenue
from gold.fact_booking
group by passenger_id
order by net_revenue desc
limit 10

In [0]:
-- Q06 top 10 passengers as a share of total revenue
with total as (
  select sum(ticket_price * (1 - discount)) as total_revenue
  from gold.fact_booking
),
by_passenger as (
  select passenger_id,
  sum(ticket_price * (1 - discount)) as passenger_revenue
  from gold.fact_booking
  group by passenger_id
)
select p.passenger_id,
round(p.passenger_revenue, 2) as net_revenue,
round(100 * p.passenger_revenue / t.total_revenue, 2) as revenue_share_pct
from by_passenger p
cross join total t
order by net_revenue desc
limit 10

In [0]:
-- Q07 top 10 routes by net revenue
select r.ro_name as route_name,
       r.ro_airline_brand as airline_brand,
       round(sum(b.ticket_price * (1 - b.discount)), 2) as net_revenue
from gold.fact_booking b
join gold.dim_rooute r
  on b.route_id = r.ro_routeid
group by r.ro_name, r.ro_airline_brand
order by net_revenue desc
limit 10 

In [0]:
-- Q08 bottom 10 routes by net revenue
select r.ro_name as route_name,
       r.ro_airline_brand as airline_brand,
       round(sum(b.ticket_price * (1 - b.discount)), 2) as net_revenue
from gold.fact_booking b
join gold.dim_rooute r
  on b.route_id = r.ro_routeid
group by r.ro_name, r.ro_airline_brand
order by net_revenue
limit 10 

In [0]:
-- Q09 revenue by route type
select r.ro_routetype as route_type,
       round(sum(b.ticket_price * (1 - b.discount)), 2) as net_revenue
from gold.fact_booking b
join gold.dim_rooute r
  on b.route_id = r.ro_routeid
group by route_type
order by net_revenue desc

In [0]:
-- Q10 top 10 airlines by revenue
select r.ro_airline_brand as airline_brand,
       round(sum(b.ticket_price * (1 - b.discount)), 2) as net_revenue
from gold.fact_booking b
join gold.dim_rooute r
  on b.route_id = r.ro_routeid
group by r.ro_airline_brand
order by net_revenue desc
limit 10

In [0]:
-- Q11 Revenue sahre of top 10 airlines
with total as (
  select sum(ticket_price * (1 - discount)) as total_revenue
  from gold.fact_booking
),
by_airline as (
  select r.ro_airline_brand as airline_brand,
         sum(b.ticket_price * (1 - b.discount)) as airline_revenue
  from gold.fact_booking b
  join gold.dim_rooute r
    on b.route_id = r.ro_routeid
  group by r.ro_airline_brand
)
select a.airline_brand,
       round(a.airline_revenue, 2) as net_revenue,
       round(100 * a.airline_revenue / t.total_revenue, 2) as revenue_share_pct
from by_airline a
cross join total t
order by net_revenue desc
limit 10

In [0]:
-- Q12 revenue by passenger region 
select g.r_name as passenger_region,
       round(sum(b.ticket_price * (1 - b.discount)), 2) as net_revenue
from gold.fact_booking b
join gold.dim_passenger p
  on b.passenger_id = p.pa_passengerid
join gold.dim_geography g
  on p.pa_nationkey = g.n_nationkey
group by passenger_region
order by net_revenue desc

In [0]:
-- Q13 revenue by passenger nation
select g.n_name as passenger_nation,
       round(sum(b.ticket_price * (1 - b.discount)), 2) as net_revenue
from gold.fact_booking b
join gold.dim_passenger p
  on b.passenger_id = p.pa_passengerid
join gold.dim_geography g
  on p.pa_nationkey = g.n_nationkey
group by passenger_nation
order by net_revenue desc

In [0]:
-- Q14 revenue and volume split by refund flag
select refund_flag,
       round(sum(ticket_price * (1 - discount)), 2) as net_revenue,
       count(distinct booking_id) as booking_volume,
       round(sum(ticket_price * (1 - discount)) / count(distinct booking_id), 2) as avg_revenue_per_booking
from gold.fact_booking
group by refund_flag
order by net_revenue desc

In [0]:
-- Q15 average discount by travel purpose
select p.pa_travelpurpose as travel_purpose,
       round(avg(b.discount), 4) as avg_discount
from gold.fact_booking b
join gold.dim_passenger p
  on b.passenger_id = p.pa_passengerid
group by travel_purpose
order by avg_discount desc

In [0]:
-- Q16 average flight duration booking departure to arrival in hours by cabin class
select cabin_class,
       round(avg(datediff(arrival_date, departure_date)), 2) as avg_flight_duration_hours
from gold.fact_booking
group by cabin_class
order by avg_flight_duration_hours desc

In [0]:
-- Q17 booking counts and revenue by booking priority
select booking_priority,
       round(sum(ticket_price * (1 - discount)), 2) as net_revenue,
       count(distinct booking_id) as booking_volume,
       round(sum(ticket_price * (1 - discount)) / count(distinct booking_id), 2) as avg_revenue_per_booking
from gold.fact_booking
group by booking_priority
order by net_revenue desc

In [0]:
-- Q18 monthly breakdown of booking status
select year(booking_date) as year,
       month(booking_date) as month,
       booking_status,
       count(distinct booking_id) as booking_volume,
       round(sum(ticket_price * (1 - discount)), 2) as net_revenue
from gold.fact_booking
group by year, month, booking_status
order by year, month, booking_status

In [0]:
-- Q19 route revenue ranking within each travel purpose
with route_revenue as (
  select p.pa_travelpurpose as travel_purpose,
         r.ro_name as route_name,
         round(sum(b.ticket_price * (1 - b.discount)), 2) as net_revenue
  from gold.fact_booking b
  join gold.dim_passenger p
    on b.passenger_id = p.pa_passengerid
  join gold.dim_rooute r
    on b.route_id = r.ro_routeid
  group by travel_purpose, route_name
)
select travel_purpose,
       route_name,
       net_revenue,
       rank() over (partition by travel_purpose order by net_revenue desc) as revenue_rank
from route_revenue
order by travel_purpose, revenue_rank

In [0]:
-- Q20 revenue comparision of passenger geography versus airline geography by region
with passenger_region_revenue as (
  select g.r_name as passenger_region,
         round(sum(b.ticket_price * (1 - b.discount)), 2) as passenger_revenue
  from gold.fact_booking b
  join gold.dim_passenger p
    on b.passenger_id = p.pa_passengerid
  join gold.dim_geography g
    on p.pa_nationkey = g.n_nationkey
  group by passenger_region
),
airline_region_revenue as (
  select g.r_name as airline_region,
         round(sum(b.ticket_price * (1 - b.discount)), 2) as airline_revenue
  from gold.fact_booking b
  join gold.dim_airline al
    on b.airline_id = al.al_airlineid
  join gold.dim_geography g
    on al.al_nationkey = g.n_nationkey
  group by airline_region
)
select coalesce(p.passenger_region, a.airline_region) as region,
       p.passenger_revenue,
       a.airline_revenue,
       round(p.passenger_revenue - a.airline_revenue, 2) as revenue_difference
from passenger_region_revenue p
full outer join airline_region_revenue a
  on p.passenger_region = a.airline_region
order by region